In [3]:
import numpy as np
from scipy.integrate import quad

# Define the Legendre polynomials
def legendre_polynomial_0(x):
    return 1

def legendre_polynomial_1(x):
    return x

def legendre_polynomial_2(x):
    return (3*x**2 - 1) / 2

def legendre_polynomial_3(x):
    return (5*x**3 - 3*x) / 2

def legendre_polynomial_4(x):
    return (35*x**4 - 30*x**2 + 3) / 8

# Define the function f(x)
def f(x):
    return np.exp(-x**2) * np.sin(x - 1) + 1

# Calculate the integrals
integrals_f = [quad(lambda x: f(x) * legendre_polynomial(x), -1, 1)[0] for legendre_polynomial in [legendre_polynomial_0, legendre_polynomial_1, legendre_polynomial_2, legendre_polynomial_3, legendre_polynomial_4]]
integrals_legendre_squared = [quad(lambda x: legendre_polynomial(x)**2, -1, 1)[0] for legendre_polynomial in [legendre_polynomial_0, legendre_polynomial_1, legendre_polynomial_2, legendre_polynomial_3, legendre_polynomial_4]]

# Calculate the coefficients
coefficients = [integral_f / integral_legendre_squared for integral_f, integral_legendre_squared in zip(integrals_f, integrals_legendre_squared)]

# Print the coefficients
for i, c in enumerate(coefficients):
    print(f"c{i}: {c}")


c0: 0.44784831278662507
c1: 0.2809149099236387
c2: 0.48389388416416174
c3: -0.1364567477339161
c4: -0.11273025713192444


In [15]:
import numpy as np
from scipy.optimize import minimize
from scipy.integrate import quad

# Define the Legendre polynomial function
def legendre_polynomial(n, x):
    if n == 0:
        return np.ones_like(x)
    elif n == 1:
        return x
    elif n == 2:
        return (3 * x**2 - 1) / 2
    elif n == 3:
        return (5 * x**3 - 3 * x) / 2
    elif n == 4:
        return (35 * x**4 - 30 * x**2 + 3) / 8

# Define the function f(x)
def f(x):
    return np.exp(-x**2) * np.sin(x - 1) + 1

def objective_function(c):
    """
    ===============================================================================
    Calculates the objective function value for a given set of coefficients (c).

    Parameters:
        c : coefficients for the Legendre polynomial approximation.

    Returns:
        The value of the objective function, representing the squared error.
    ================================================================================
    """
    n = len(c)
    integral_terms = np.zeros(n)
    
    for i in range(n):
        terms = np.array([c[j] * legendre_polynomial(j, x) for j in range(n)]) 
        integral_terms[i] = np.trapz(f(x) - np.sum(terms, axis=0), x) ** 2
    
    return np.sum(integral_terms)

def gradient(c):
    '''
    ======================================================================
    Calculates the gradient of the objective function W
    
    Parameters:
        c : coefficients for the Legendre polynomial approximation. 
    ======================================================================
    '''
    n = len(c)
    grad = np.zeros(n)
    for i in range(n):
        integral_term = np.trapz(f(x) * legendre_polynomial(i, x), x) - 2 * c[i] / (2 * i + 1)
        grad[i] = -2 * integral_term
    return grad


# Domain for plotting
x = np.linspace(-1, 1, 100)

# Initial guess for coefficients c_i
max_n = 4
initial_guess = np.zeros(max_n+1)

# Minimize W using gradients
result = minimize(objective_function, initial_guess, jac=gradient, method='BFGS')

# Extract the optimized coefficients
optimal_c = result.x
print("-"*30)
print("Optimized coefficients:")
print("-"*30)

for n in range(max_n+1):
    print(f"c{n}: {optimal_c[n]}")
print("-"*30)


------------------------------
Optimized coefficients:
------------------------------
c0: 0.4478686737695029
c1: 0.2809144444406612
c2: 0.4844153372676482
c3: -0.13626722666974048
c4: -0.10999294494482652
------------------------------


In [16]:
import numpy as np
from scipy.optimize import minimize

# Define the Legendre polynomial function
def legendre_polynomial(n, x):
    if n == 0:
        return np.ones_like(x)
    elif n == 1:
        return x
    elif n == 2:
        return (3 * x**2 - 1) / 2
    elif n == 3:
        return (5 * x**3 - 3 * x) / 2
    elif n == 4:
        return (35 * x**4 - 30 * x**2 + 3) / 8

# Define the function f(x)
def f(x):
    return np.exp(-x**2) * np.sin(x - 1) + 1

# Define the objective function
def objective_function(c):
    n = len(c)
    integral_terms = np.zeros(n)
    
    for i in range(n):
        terms = np.array([c[j] * legendre_polynomial(j, x) for j in range(n)]) 
        integral_terms[i] = np.trapz(f(x) - np.sum(terms, axis=0), x) ** 2
    
    return np.sum(integral_terms)

# Define the gradient function using complex step differentiation
def gradient_complex_step(c):
    h = 1e-12
    n = len(c)
    grad = np.zeros(n)
    
    for i in range(n):
        c_complex0 = c.copy()
        c_complex = np.complex(c_complex0, h) 
        grad[i] = np.imag(objective_function(c_complex)) / h
    
    return grad

# Domain for plotting
x = np.linspace(-1, 1, 100)

# Initial guess for coefficients c_i
max_n = 4
initial_guess = np.zeros(max_n+1)

# Minimize W using gradients computed with complex step differentiation
result = minimize(objective_function, initial_guess, jac=gradient_complex_step, method='BFGS')

# Extract the optimized coefficients
optimal_c = result.x

print("-"*30)
print("Optimized coefficients:")
print("-"*30)

for n in range(max_n+1):
    print(f"c{n}: {optimal_c[n]}")
print("-"*30)


------------------------------
Optimized coefficients:
------------------------------
c0: 0.0
c1: 0.0
c2: 0.0
c3: 0.0
c4: 0.0
------------------------------


/tmp/ipykernel_1346008/2051891823.py:40: ComplexWarning: Casting complex values to real discards the imaginary part
  c_complex[i] += 1j * h
